# AgentBlame / DCFA on Kaggle

Before running: right panel -> **Add Input** -> your `dcfa` dataset; **Accelerator** -> GPU; **Internet** -> On.

Run the cells top to bottom (Shift+Enter).

In [ ]:
%%bash
# STEP 1: copy the code to /kaggle/temp. Only results, cache and logs live in /kaggle/working (saved as output),
# because Kaggle refuses file names with '&' (the Who&When folder is uploaded as Who_and_When).
SRC=$(dirname "$(find /kaggle/input -name run.py -path '*DCFA*' | head -1)")
if [ "$SRC" = "." ]; then echo "ERROR: dcfa dataset not found. Right panel -> Add Input -> your dcfa dataset"; exit 1; fi
CODE=/kaggle/temp/DCFA
rm -rf "$CODE" && mkdir -p "$CODE" && cp -r "$SRC"/. "$CODE"/
mv "$CODE/Who_and_When" "$CODE/Who&When"
mkdir -p /kaggle/working/results /kaggle/working/results_smoke /kaggle/working/results_check /kaggle/working/logs /kaggle/working/.cache
# The uploaded .cache (answers made on the Mac) is NOT used: every answer is made on this GPU, so all runs share one setup
# Resume: if a previous run of this notebook is attached as input, bring back its results, cache and logs
for d in $(find /kaggle/input -maxdepth 3 -type d -name results); do
  P=$(dirname "$d"); for x in results results_smoke results_check logs .cache; do [ -d "$P/$x" ] && cp -rn "$P/$x"/. /kaggle/working/$x/; done
  echo "Restored previous results from $P"
done
rm -rf "$CODE/.cache"
for x in results results_smoke results_check logs .cache; do ln -sfn /kaggle/working/$x "$CODE/$x"; done
echo "Code ready in $CODE"; ls "$CODE"

In [ ]:
%%bash
# STEP 2: install Ollama, start it, download the model (~5-10 min)
nvidia-smi --query-gpu=name,memory.total --format=csv || { echo "ERROR: no GPU. Right panel -> Accelerator -> GPU"; exit 1; }
curl -sI --max-time 15 https://ollama.com > /dev/null || { echo "ERROR: no internet. Settings -> Turn on Internet (needs phone verification), then run this cell again"; exit 1; }
set -o pipefail
apt-get install -y -qq zstd pciutils > /dev/null 2>&1
curl -fsSL https://ollama.com/install.sh | sh > /dev/null || { echo "ERROR: Ollama install failed"; exit 1; }
pgrep -x ollama > /dev/null || { echo "Starting Ollama"; OLLAMA_NUM_PARALLEL=1 OLLAMA_KEEP_ALIVE=-1 setsid nohup ollama serve >> /kaggle/working/logs/ollama.log 2>&1 < /dev/null & sleep 5; }
cd /kaggle/temp/DCFA && set -a && source .env && set +a
ollama pull "$OLLAMA_MODEL" 2>&1 | tail -1 || { echo "ERROR: model download failed"; exit 1; }
ollama pull "$OLLAMA_EMBED_MODEL" 2>&1 | tail -1 || { echo "ERROR: embedding model download failed"; exit 1; }
pip install -q python-dotenv tqdm
python llm.py
ollama ps   # PROCESSOR should say 100% GPU

In [ ]:
%%bash
# STEP 3: CHECK. One short log from every dataset (2 originals + 12 perturbed), all 4 methods. A few minutes.
# Start Ollama if it is not running (e.g. after a stopped cell)
pgrep -x ollama > /dev/null || { echo "Starting Ollama"; OLLAMA_NUM_PARALLEL=1 OLLAMA_KEEP_ALIVE=-1 setsid nohup ollama serve >> /kaggle/working/logs/ollama.log 2>&1 < /dev/null & sleep 5; }
set -o pipefail  # stop if run.py fails, even though its output is filtered
cd /kaggle/temp/DCFA
M=all_at_once,step_by_step,binary_search,dcfa
for d in "Who&When/Algorithm-Generated" perturbed/Algorithm-Generated__*; do
  echo "== $d"; python run.py --data "$d" --method $M --files 38.json --out /kaggle/working/results_check 2>&1 | grep -v '%|' || exit 1
done
for d in "Who&When/Hand-Crafted" perturbed/Hand-Crafted__*; do
  echo "== $d"; python run.py --data "$d" --method $M --files 24.json --out /kaggle/working/results_check 2>&1 | grep -v '%|' || exit 1
done

In [ ]:
# STEP 3b: table of the check results. Look for: errors, DCFA fallbacks, 'Unknown' agents, and the seconds column.
import glob, json, os
print(f"{'dataset':36} {'method':14} {'file':8} {'truth':22} {'predicted':22} {'sec':>6}  problems")
for path in sorted(glob.glob('/kaggle/working/results_check/*/*/*.jsonl')):
    dataset, method = path.split('/')[-2], os.path.basename(path)[:-6]
    for r in map(json.loads, open(path)):
        problems = r.get('error', '') or r.get('trace', {}).get('fallback', '')
        print(f"{dataset:36} {method:14} {r['file']:8} {str(r['gt_agent']) + ' @' + str(r['gt_step']):22} {str(r['agent']) + ' @' + str(r['step']):22} {r.get('seconds', '?'):>6}  {problems}")

In [ ]:
%%bash
# STEP 3c (optional, slow): the longest logs, to measure speed and check the 128k context. Can take 1-2+ hours.
# Start Ollama if it is not running (e.g. after a stopped cell)
pgrep -x ollama > /dev/null || { echo "Starting Ollama"; OLLAMA_NUM_PARALLEL=1 OLLAMA_KEEP_ALIVE=-1 setsid nohup ollama serve >> /kaggle/working/logs/ollama.log 2>&1 < /dev/null & sleep 5; }
cd /kaggle/temp/DCFA && PY=python bash run_all.sh smoke

In [ ]:
%%bash
# STEP 4: the full run. Stops itself after 8h so Kaggle saves the results before its time limit.
# Start Ollama if it is not running (e.g. after a stopped cell)
pgrep -x ollama > /dev/null || { echo "Starting Ollama"; OLLAMA_NUM_PARALLEL=1 OLLAMA_KEEP_ALIVE=-1 setsid nohup ollama serve >> /kaggle/working/logs/ollama.log 2>&1 < /dev/null & sleep 5; }
cd /kaggle/temp/DCFA && timeout -s INT 8h env PY=python bash run_all.sh
tail -5 logs/progress.txt

In [ ]:
%%bash
# DOWNLOAD: pack all results into one zip -> right panel 'Output' -> all_results.zip -> Download
cd /kaggle/working && rm -f all_results.zip && zip -rq all_results.zip results results_check results_smoke logs && ls -lh all_results.zip